In [ ]:
#%cd ..

c:\Users\coffe\Desktop\titanic-ml\titanic-ml


In [3]:
import numpy as np
import pandas as pd

from omegaconf import OmegaConf

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from src.models import get_model
from src.pipeline_utils import get_preprocessor

In [6]:
config = OmegaConf.load('config/config.yaml')

train = pd.read_csv(config.paths.train_path)
test = pd.read_csv(config.paths.test_path)

X = train.drop(columns='Survived')
y = train['Survived']

model_cfg = config.models.lightgbm

model_params = OmegaConf.to_container(
    model_cfg.params,
    resolve=True,
)

## 1-fold experiment

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=config.general.seed,
)

preprocessor = get_preprocessor(
    model_cfg.model_type,
    config,
)

model = get_model(
    model_name='lightgbm',
    seed=config.general.seed,
    **model_params,
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model),
])

pipeline.fit(X_train, y_train)

val_predictions = pipeline.predict(X_val)

val_accuracy = accuracy_score(
    y_val,
    val_predictions,
)

print(f"1-fold validation accuracy: {val_accuracy:.4f}")

1-fold validation accuracy: 0.7989


In [11]:
test_proba_1_fold = pipeline.predict_proba(test)[:, 1]

test_pred_1_fold = (test_proba_1_fold >= config.general.threshold).astype(int)

submission_1_fold = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_pred_1_fold,
})

submission_1_fold.to_csv(
    "submission_lgbm_1_fold.csv",
    index=False,
)

## 5-fold experiment

In [12]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=config.general.seed,
)

fold_scores = []
test_fold_probs = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    
    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]
    
    preprocessor = get_preprocessor(
        model_cfg.model_type,
        config,
    )
    
    model = get_model(
        model_name='lightgbm',
        seed=config.general.seed,
        **model_params
    )
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model),
    ])
    
    pipeline.fit(X_train, y_train)
    
    val_predictions = pipeline.predict(X_val)
    
    score = accuracy_score(y_val, val_predictions)
    
    fold_scores.append(score)
    
    test_proba = pipeline.predict_proba(test)[:, 1]
    
    test_fold_probs.append(test_proba)
    
    print(
        f"Fold {fold}: "
        f"accuracy = {score:.4f}"
    )
    
fold_scores = np.array(fold_scores)

print(
    f"\n5-fold CV accuracy: "
    f"{fold_scores.mean():.4f} ± "
    f"{fold_scores.std():.4f}"
)

mean_test_proba = np.mean(test_fold_probs, axis=0)

test_pred_5_fold = (mean_test_proba >= config.general.threshold).astype(int)

submission_5_fold = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_pred_5_fold,
})

submission_5_fold.to_csv(
    "submission_lgbm_5_fold.csv",
    index=False,
)

Fold 1: accuracy = 0.8715
Fold 2: accuracy = 0.8652
Fold 3: accuracy = 0.7921
Fold 4: accuracy = 0.8483
Fold 5: accuracy = 0.8652

5-fold CV accuracy: 0.8485 ± 0.0292


In [13]:
comparison = pd.DataFrame({
    "experiment": [
        "1-fold",
        "5-fold",
    ],
    "local_accuracy": [
        val_accuracy,
        fold_scores.mean(),
    ],
})

comparison

,experiment,local_accuracy
0,1-fold,0.798883
1,5-fold,0.848459


In [14]:
comparison["kaggle_score"] = [
    0.76794,
    0.75119,
]

comparison

,experiment,local_accuracy,kaggle_score
0,1-fold,0.798883,0.76794
1,5-fold,0.848459,0.75119


## Conclusion

The 5-fold approach achieved a much higher local validation accuracy than the single-split approach.

However, the Kaggle public leaderboard score was higher for the 1-fold experiment:

- 1-fold: local accuracy = 0.7989, Kaggle score = 0.76794
- 5-fold: local accuracy = 0.8485, Kaggle score = 0.75119

This experiment shows that a higher cross-validation score does not always translate directly into a higher public leaderboard score.

Possible reasons include the small dataset size, differences between local validation splits and the hidden Kaggle test distribution, and variance in the public leaderboard subset.